In [ ]:
#!git clone https://github.com/whyhardt/SPICE.git

In [ ]:
# !pip install -e SPICE

In [23]:
import sys

import torch

from spice import SpiceEstimator

sys.path.append("../../..")
from weinhardt2026.utils.benchmarking_gru import GRUModel, training
from weinhardt2026.studies.castro2025.benchmarking_castro2025 import Castro2025Model, get_dataset, generate_behavior
import spice_castro2025, spice_castro2025_2

## Load dataset

In [24]:
path_data = 'data/eckstein2024.csv'
test_sessions = (2,)  # pick sessions that exist for all participants; adjust if needed
dataset_train, dataset_test, info_dataset = get_dataset(path_data=path_data, test_sessions=test_sessions, verbose=True)

Shape of dataset: torch.Size([4158, 150, 1, 13])
Number of participants: 862
Number of actions in dataset: 4


In [3]:
# keep only 100 participants for rapid prototyping
from spice import SpiceDataset

keep_participants = torch.arange(0, 100)

mask_train = torch.isin(dataset_train.xs[..., -1], keep_participants)
mask_test = torch.isin(dataset_test.xs[..., -1], keep_participants)

xs, ys = dataset_train.xs[mask_train], dataset_train.ys[mask_train]
dataset_train = SpiceDataset(xs, ys)

xs, ys = dataset_test.xs[mask_test], dataset_test.ys[mask_test]
dataset_test = SpiceDataset(xs, ys)


## SPICE Setup

## SPICE Training

Let's setup now the `SpiceEstimator` object and fit it to the data! 

We are going to do this in two steps:

1. Without fitting the SINDy coefficients to get the pure RNN performance given the selected architecture. 
2. With fitting SINDy coefficients to get the final performance of the interpretable model

That way we can disentangle the gap between GRU and SPICE w.r.t. architecture and SINDy library 

In [ ]:
from spice import SpiceConfig


CONFIG = SpiceConfig(
    library_setup={
        'value_reward_env': [
            'reward',
        ],
        'value_reward_chosen': [
            'reward_env',
            'reward',
            'dvalue',
            'd²value',
        ],
        'value_reward_not_chosen': [
            'reward_env',
            'dvalue',
            'd²value',
        ],
        'value_choice_chosen': [
            'dvalue',
            'd²value',
        ],
        'value_choice_not_chosen': [
            'dvalue',
            'd²value',
        ],
    },
    memory_state={
        'value_reward_env': 0.,
        
        'value_reward': 0.,
        'dvalue_reward': 0.,
        'd²value_reward': 0.,
        'buffer_value_reward': 0,
        'buffer_dvalue_reward': 0.,
        
        'value_choice': 0.,
        'dvalue_choice': 0.,
        'd²value_choice': 0.,
        'buffer_value_choice': 0,
        'buffer_dvalue_choice': 0.,
    },
    states_in_logit=['value_reward', 'value_choice'],
)

In [ ]:
from spice import BaseModel


class SpiceModel(BaseModel):
    """
    Value learning with delta-value working memory buffers.
    
    Buffers store recent value changes (dV) rather than raw observations or values.
    The current state encodes the level; dV encodes the recent trajectory
    (trend, volatility), providing non-redundant temporal information.
    """
    
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        
        dropout = 0.1
        
        self.participant_embedding = self.setup_embedding(self.n_participants, self.embedding_size, dropout=dropout)
        
        self.setup_module(key_module='value_reward_env', input_size=1+self.embedding_size, dropout=dropout)
        self.setup_module(key_module='value_reward_chosen', input_size=4+self.embedding_size, dropout=dropout)
        self.setup_module(key_module='value_reward_not_chosen', input_size=3+self.embedding_size, dropout=dropout)
        self.setup_module(key_module='value_choice_chosen', input_size=2+self.embedding_size, dropout=dropout)
        self.setup_module(key_module='value_choice_not_chosen', input_size=2+self.embedding_size, dropout=dropout)
        
    def forward(self, inputs, state=None):
        spice_signals = self.init_forward_pass(inputs, state)

        reward_full = spice_signals.rewards.sum(dim=-1, keepdim=True).expand_as(spice_signals.actions)
        
        participant_embedding = self.participant_embedding(spice_signals.participant_ids)

        for trial in spice_signals.trials:
            
            # Snapshot pre-update values for delta computation
            self.state['buffer_value_reward'] = self.get_state()['value_reward']
            self.state['buffer_value_choice'] = self.get_state()['value_choice']
            self.state['buffer_dvalue_reward'] = self.get_state()['dvalue_reward']
            self.state['buffer_dvalue_choice'] = self.get_state()['dvalue_choice']
            
            # REWARD VALUE UPDATES
            value_reward_env = self.call_module(
                key_module='value_reward_env',
                key_state='value_reward_env',
                inputs=reward_full[trial],
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embedding,
            )
            
            self.call_module(
                key_module='value_reward_chosen',
                key_state='value_reward',
                action_mask=spice_signals.actions[trial],
                inputs=(
                    value_reward_env,
                    spice_signals.rewards[trial],
                    self.state['dvalue_reward'],
                    self.state['d²value_reward'],
                ),
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embedding,
            )

            self.call_module(
                key_module='value_reward_not_chosen',
                key_state='value_reward',
                action_mask=1-spice_signals.actions[trial],
                inputs=(
                    value_reward_env,
                    self.state['dvalue_reward'],
                    self.state['d²value_reward'],
                ),
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embedding,
            )

            # CHOICE VALUE UPDATES
            self.call_module(
                key_module='value_choice_chosen',
                key_state='value_choice',
                action_mask=spice_signals.actions[trial],
                inputs=(
                    self.state['dvalue_choice'],
                    self.state['d²value_choice'],
                ),
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embedding,
            )
            
            self.call_module(
                key_module='value_choice_not_chosen',
                key_state='value_choice',
                action_mask=1-spice_signals.actions[trial],
                inputs=(
                    self.state['dvalue_choice'],
                    self.state['d²value_choice'],
                ),
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embedding,
            )
            
            # BUFFER UPDATES: compute deltas and shift
            self.state['dvalue_reward'] = self.state['value_reward'] - self.state['buffer_value_reward']
            self.state['dvalue_choice'] = self.state['value_choice'] - self.state['buffer_value_choice']
            self.state['d²value_reward'] = self.state['dvalue_reward'] - self.state['buffer_dvalue_reward']
            self.state['d²value_choice'] = self.state['dvalue_choice'] - self.state['buffer_dvalue_choice']

            # Logits
            spice_signals.logits[trial] = (
                self.state['value_reward'] 
                + self.state['value_choice']
            )

        spice_signals = self.post_forward_pass(spice_signals)
        return spice_signals.logits, self.get_state()

In [ ]:
# fitting without SINDy coefficients
path_spice = 'params/spice_castro2025.pkl'

estimator = SpiceEstimator(
        # model paramaeters
        spice_class=SpiceModel,
        spice_config=CONFIG,
        n_actions=dataset_train.n_actions,
        n_participants=dataset_train.n_participants,
        
        epochs=1000,
        
        sindy_weight=0,
        sindy_library_polynomial_degree=2,
        sindy_pruning_frequency=100,
        sindy_ensemble_pruning=0.05,
        sindy_threshold_pruning=0.01,
        sindy_alpha=0.0001,
        
        save_path_spice=path_spice,
        verbose=True,
    )
estimator.fit(dataset_train.xs, dataset_train.ys, dataset_test.xs, dataset_test.ys)

In [25]:
path_spice = 'params/spice_castro2025.pkl'

estimator = SpiceEstimator(
        # model paramaeters
        spice_class=spice_castro2025_2.SpiceModel,
        spice_config=spice_castro2025_2.CONFIG,
        n_actions=dataset_train.n_actions,
        n_participants=dataset_train.n_participants,
        
        epochs=10,
        warmup_steps=200,
        learning_rate=0.01,
        ensemble_size=10,
        
        sindy_weight=0.1,
        sindy_alpha=0.0001,
        sindy_pruning_frequency=100,
        sindy_threshold_pruning=0.01,
        sindy_ensemble_pruning=0.05,
        sindy_library_polynomial_degree=2,

        verbose=True,
        # device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
        save_path_spice=path_spice,
    )

In [ ]:
estimator.fit(dataset_train.xs, dataset_train.ys, dataset_test.xs, dataset_test.ys)

In [19]:
estimator.load_spice(path_spice)
estimator.aggregate_coefficients()

In [20]:
# Print example SPICE model for first participant
print("\nExample SPICE model (participant 0):")
estimator.print_spice_model(participant_id=5)


Example SPICE model (participant 0):
value_reward_env[t+1]        = 1.0 value_reward_env[t] 
value_reward_chosen[t+1]     = 1.0 value_reward_chosen[t] 
value_reward_not_chosen[t+1] = 1.0 value_reward_not_chosen[t] 
value_choice_chosen[t+1]     = 1.0 value_choice_chosen[t] 
value_choice_not_chosen[t+1] = 1.0 value_choice_not_chosen[t] 
volatility_chosen[t+1]       = 1.0 volatility_chosen[t] 
volatility_not_chosen[t+1]   = 1.0 volatility_not_chosen[t] 


## Benchmarking

### Castro2025 benchmark model

In [26]:
# Benchmark model: Castro et al. 2025
cfs = Castro2025Model(
    n_participants=dataset_train.n_participants,
    n_actions=dataset_train.n_actions,
    batch_first=True,
    )

path_cfs = path_spice.replace('spice_', 'cfs_')

In [ ]:
optimizer_cfs = torch.optim.Adam(params=cfs.parameters(), lr=0.01)
epochs = 1000

cfs = training(
    model=cfs,
    optimizer=optimizer_cfs,
    dataset_train=dataset_train,
    dataset_test=dataset_test,
    epochs=epochs,
)

torch.save(cfs.state_dict(), path_cfs)

In [28]:
cfs.load_state_dict(torch.load(path_cfs, map_location='cpu'))

<All keys matched successfully>

### GRU Model

In [29]:
gru = GRUModel(
    n_actions=dataset_train.n_actions, 
    additional_inputs=2, 
    dropout=0.1,
    hidden_size=32,
    )
path_gru = path_spice.replace('spice_', 'gru_')

In [30]:
epochs = 1000
optimizer_gru = torch.optim.Adam(gru.parameters(), lr=0.01)

gru = training(
    model=gru,
    optimizer=optimizer_gru,
    dataset_train=dataset_train,
    dataset_test=dataset_test,
    epochs=epochs,
    ).to(torch.device('cpu'))

torch.save(gru.state_dict(), path_gru)

Epoch 1/1000: L(Train): 1.4832992553710938; L(Test): 1.4389615058898926
Epoch 2/1000: L(Train): 1.46689772605896; L(Test): 1.413777470588684
Epoch 3/1000: L(Train): 1.4281771183013916; L(Test): 1.4077895879745483
Epoch 4/1000: L(Train): 1.417781114578247; L(Test): 1.3911715745925903
Epoch 5/1000: L(Train): 1.4015998840332031; L(Test): 1.3812556266784668
Epoch 6/1000: L(Train): 1.3912235498428345; L(Test): 1.3835245370864868
Epoch 7/1000: L(Train): 1.3939082622528076; L(Test): 1.3854774236679077
Epoch 8/1000: L(Train): 1.3951915502548218; L(Test): 1.3808187246322632
Epoch 9/1000: L(Train): 1.390014886856079; L(Test): 1.3734103441238403
Epoch 10/1000: L(Train): 1.3810099363327026; L(Test): 1.3690599203109741
Epoch 11/1000: L(Train): 1.3759630918502808; L(Test): 1.3678592443466187
Epoch 12/1000: L(Train): 1.3748602867126465; L(Test): 1.3648086786270142
Epoch 13/1000: L(Train): 1.3717060089111328; L(Test): 1.3582593202590942
Epoch 14/1000: L(Train): 1.3641979694366455; L(Test): 1.349668502

In [10]:
gru.load_state_dict(torch.load(path_gru, map_location='cpu'))

<All keys matched successfully>

# ANALYSIS

In [31]:
from weinhardt2026.analysis.analysis_model_evaluation import analysis_model_evaluation
from weinhardt2026.analysis.analysis_coefficients_distributions import analysis_coefficients_distributions
from weinhardt2026.analysis.analysis_coefficients_individuals import analysis_coefficients_individuals
from analysis_generative import analysis_generative_behavior

## General Analysis

In [ ]:
# Dataset-specific behavioral analysis placeholder.
# Replace with Eckstein2024-specific columns if needed.

In [ ]:
print("Fitted Castro2025 parameters:")
print("\nBeta_r")
print(cfs.beta_r)
print("\nLapse")
print(cfs.lapse)
print("\nPrior")
print(cfs.prior)
print("\nAlpha exploration rate")
print(cfs.alpha_exploration_rate)
print("\nDecay rate")
print(cfs.decay_rate)
print("\nGamma")
print(cfs.gamma)
print("\nTemperature")
print(cfs.temperature)

## Analysis Model Evaluation

In [ ]:
analysis_model_evaluation(
    dataset=dataset_test,
    spice_model=estimator,
    benchmark_model=cfs.to(torch.device('cpu')),
    gru_model=gru.eval().to(torch.device('cpu')),
    )

Computing choice probabilities with benchmark model...
Computing choice probabilities with GRU model...
Computing choice probabilities with SPICE model...


,Trial Lik.,(std),n_parameters,(std),NLL,AIC,BIC
Benchmark,0.621020,0.159128,13.0,0.0,232753.890625,4.655338e+05,4.656781e+05
GRU,0.620423,0.157723,6852.0,0.0,233223.875000,4.801518e+05,5.562038e+05
SPICE-RNN,0.155207,0.123891,101922.0,0.0,910215.437500,2.024275e+06,3.155533e+06
SPICE,0.249915,0.005205,37.0,0.0,677476.187500,1.355026e+06,1.355437e+06


## Analysis coefficient distributions

In [ ]:
analysis_coefficients_distributions(
    spice_model=estimator,
    output_dir='results',
)

## Analysis Individual Differences

In [ ]:
# analysis_coefficients_individuals(
#     criterion="SomeCriterionColumnInYourDataset",
#     analysis="disc",  # also: "cont"
#     reference="ReferenceGroupFromCriterionColumn",  # only necessary if analysis="disc"
    
#     path_data=path_file,
    
#     spice_model=estimator,
    
#     dir_output='results',
# )

## Generative Behavior Analysis

In [ ]:
estimator.eval()
generate_behavior(
    model=estimator,
    dataset=dataset_train,
    save_dataset='data/eckstein2024_spice.csv'
)

estimator.use_sindy(False)
generate_behavior(
    model=estimator,
    dataset=dataset_train,
    save_dataset='data/eckstein2024_spice_rnn.csv'
)

gru.eval()
generate_behavior(
    model=gru,
    dataset=dataset_train,
    save_dataset='data/eckstein2024_gru.csv'
)

generate_behavior(
    model=cfs,
    dataset=dataset_train,
    save_dataset='data/eckstein2024_cfs.csv'
)

In [ ]:
analysis_generative_behavior(
    path_data_real=path_data,
    path_data_gru='data/eckstein2024_gru.csv',
    path_data_benchmark='data/eckstein2024_cfs.csv',
    path_data_spice='data/eckstein2024_spice.csv',
    path_data_spice_rnn='data/eckstein2024_spice_rnn.csv',
    output_dir='results',
)